## NFL Notebook

### Setup

In [17]:
import pandas as pd
import requests
from pathlib import PurePath
import os
import subprocess
import random
import json
from multiprocessing.dummy import Pool as ThreadPool
from multiprocessing import cpu_count
import pickle

ips = [
    "100.91.198.95",
    "100.65.216.68",
    "100.65.216.68",
    "100.70.240.117",
    "100.117.126.96",
    "100.88.22.25",
    "100.100.169.122",
    "100.79.65.118",
    "100.120.7.76",
    "100.85.149.103",
    "100.85.149.103",
    "100.66.247.50",
    "100.98.0.17",
    "100.112.151.82",
    "100.88.213.131",
    "100.112.151.82",
    "100.111.245.115",
    "100.85.34.142",
    "100.100.203.20",
    "100.81.101.39",
    "100.99.12.129",
    "100.114.228.97",
    "100.91.160.150",
    "100.66.11.120",
    "100.127.234.115",
    "100.117.20.17",
    "100.122.231.14",
    "100.114.187.107",
    "100.122.231.14",
    "100.120.246.18",
    "100.78.4.15",
    "100.123.112.109",
    "100.78.4.15",
    "100.120.246.42",
    "100.73.33.78",
    "100.127.156.146",
    "100.114.248.6",
    "100.119.231.71",
    "100.117.68.90",
    "100.112.80.91",
    "100.113.57.90",
    "100.113.57.90",
    "100.86.133.93",
    "100.100.131.39",
    "100.81.28.91",
    "100.100.131.39",
    "100.98.10.95",
    "100.109.204.162",
    "100.73.22.4",
    "100.123.7.85",
    "100.85.38.61",
    "100.118.155.102",
    "100.87.93.9",
    "100.118.155.102",
    "100.124.55.8",
    "100.114.20.52",
    "100.103.142.113",
    "100.81.170.137",
    "100.120.181.133",
    "100.108.32.59",
    "100.110.81.102",
    "100.112.134.92",
    "100.97.155.16",
    "100.120.39.100",
    "100.123.96.89",
    "100.123.96.89",
    "100.107.199.102",
    "100.103.10.58",
    "100.84.2.120",
    "100.97.43.47",
    "100.66.72.110",
    "100.84.2.120",
    "100.74.206.23",
    "100.68.203.31",
    "100.83.81.112",
    "100.80.218.70",
    "100.80.218.70",
    "100.69.91.134",
    "100.104.54.116",
    "100.117.167.110",
    "100.65.194.122",
    "100.126.111.61",
    "100.85.243.129",
    "100.117.167.110",
    "100.70.74.121",
    "100.99.135.32",
    "100.100.52.113",
    "100.82.151.77",
    "100.95.200.37",
    "100.106.242.19",
    "100.84.251.68",
    "100.82.221.88",
    "100.64.17.108",
    "100.86.172.69",
    "100.119.10.123",
    "100.91.152.42",
    "100.96.176.46",
    "100.95.30.22",
    "100.115.163.94",
    "100.93.242.75",
]

header = {
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36"
}


BASE_URL = "https://www.pro-football-reference.com/years/"
TOP_LEVEL_DIR = PurePath(os.getcwd())

RAW_DATA_DIR = PurePath(TOP_LEVEL_DIR / "nfl-data")
CLEANED_DATA_DIR = PurePath(TOP_LEVEL_DIR / "cleaned-data")

if not os.path.exists(RAW_DATA_DIR):
    os.mkdir(RAW_DATA_DIR)

stat_categories = (
    "passing",
    "rushing",
    "receiving",
    "defense",
    "kicking",
    "punting",
    "returns",
)

### Getting the data

In [ ]:
for year in range(2004, 2025):
    curr_year_path = PurePath(RAW_DATA_DIR / str(year))
    if not os.path.exists(curr_year_path):
        os.makedirs(curr_year_path)

    for category in stat_categories:
        print(f"Getting {category} stats for {year}")
        category_path = PurePath(curr_year_path / f"{category}.csv")
        url = BASE_URL + f"{year}/{category}.htm"
        response = requests.get(url, headers=header)

        if response.status_code == 200:
            try:
                df = pd.read_html(response.text)[0]
                df.to_csv(
                    category_path,
                    index=False,
                )
                print(f"Saved {year}/{category}.csv\n")
            except Exception as e:
                print(f"Error parsing {year}/{category}.csv: {e}")
        else:
            while True:
                ip = random.choice(ips)
                print(f"Request failed, trying again with {ip}")
                subprocess.run(["tailscale", "set", f"--exit-node={ip}"])
                response = requests.get(url, headers=header)

                if response.status_code == 200:
                    try:
                        df = pd.read_html(response.text)[0]
                        df.to_csv(
                            category_path,
                            index=False,
                        )
                        print(f"Saved {year}/{category}.csv\n")
                        break
                    except Exception as e:
                        print(f"Error parsing {year}/{category}.csv: {e}")

### Cleaning it up

In [42]:
valid_directories = [
    directory for directory in os.listdir(RAW_DATA_DIR) if directory != ".DS_Store"
]


def work(directory):
    for category in stat_categories:
        file_path = PurePath(RAW_DATA_DIR / directory / f"{category}.csv")
        modified_file_path = PurePath(
            RAW_DATA_DIR / directory / f"{category}_modified.csv"
        )
        df = pd.read_csv(file_path)
        if any(["Unnamed" in header for header in df.columns]):
            df.columns = df.iloc[0]
            df = df[1:]
        df = df.query("Team != '2TM'")
        df = df.drop("Rk", axis=1)
        df = df.drop("Awards", axis=1)

        unique_players = [
            player
            for player in df["Player"]
            if df["Player"].tolist().count(player) == 1
        ]
        df = df[df["Player"].isin(unique_players)]
        df = df.query("Player != 'League Average'")

        df.to_csv(modified_file_path, index=False)


def driver_function():
    with ThreadPool(processes=cpu_count()) as pool:
        pool.map(work, valid_directories)


driver_function()

### Adding the player id for players also in the college dataset

In [41]:
college_data = json.load(open(CLEANED_DATA_DIR / "cfb_player_data.json", "r"))
data_to_transfer = {
    "RAW_DATA_DIR": RAW_DATA_DIR,
    "CLEANED_DATA_DIR": CLEANED_DATA_DIR,
    "TOP_LEVEL_DIR": TOP_LEVEL_DIR,
    "stat_categories": stat_categories,
    "valid_directories": valid_directories,
    "college_data": college_data,
}
pickle.dump(
    data_to_transfer, open(PurePath(TOP_LEVEL_DIR / "nfl_data_to_transfer.pkl"), "wb")
)

In [43]:
!python nfl-data-add-pids.py

print("Player ID mapping complete")

Player ID mapping complete


### Aggregating player stats for multiple years and keeping track of it

In [55]:
!python nfl-data-combine.py

print("Data combined")

Data combined
